<a href="https://colab.research.google.com/github/anshikaguptacs24-del/AI_LAB/blob/main/LAB2_8PUZZLE_1BF24CS058.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from collections import deque

# ============================================================
# 8-PUZZLE
# BFS, DFS AND IDS
# ============================================================

# 0 represents the blank space

START = (
    5, 4, 0,
    6, 1, 8,
    7, 3, 2
)

GOAL = (
    0, 1, 2,
    3, 4, 5,
    6, 7, 8
)


# ============================================================
# PRINT THE PUZZLE
# ============================================================

def print_state(state):
    """Print a 3 x 3 puzzle board."""

    for i in range(0, 9, 3):
        row = state[i:i + 3]

        # Display 0 as blank
        print(" ".join("□" if x == 0 else str(x) for x in row))

    print()


# ============================================================
# GENERATE LEGAL MOVES
# ============================================================

def get_neighbors(state):
    """
    Generate all legal moves of the blank.

    The blank can move:
        UP
        DOWN
        LEFT
        RIGHT
    """

    # Find position of blank
    blank = state.index(0)

    # Convert position into row and column
    row = blank // 3
    col = blank % 3

    # Possible blank movements
    directions = [
        (-1, 0, "UP"),
        (1, 0, "DOWN"),
        (0, -1, "LEFT"),
        (0, 1, "RIGHT")
    ]

    neighbors = []

    for dr, dc, move in directions:

        new_row = row + dr
        new_col = col + dc

        # Check whether the blank stays inside the puzzle
        if 0 <= new_row < 3 and 0 <= new_col < 3:

            # Convert row and column back to position
            new_blank = new_row * 3 + new_col

            # Make a copy of the current state
            new_state = list(state)

            # Swap blank with adjacent tile
            new_state[blank], new_state[new_blank] = \
                new_state[new_blank], new_state[blank]

            neighbors.append(
                (tuple(new_state), move)
            )

    return neighbors


# ============================================================
# RECONSTRUCT SOLUTION PATH
# ============================================================

def reconstruct_path(parent, move_used, goal):
    """
    Starting from GOAL, move backwards using parent[]
    until START is reached.
    """

    path = []

    current = goal

    while current is not None:

        path.append(
            (current, move_used[current])
        )

        current = parent[current]

    # Currently path is:
    # GOAL -> ... -> START
    #
    # Reverse it:
    # START -> ... -> GOAL

    path.reverse()

    return path


# ============================================================
# BFS - BREADTH FIRST SEARCH
# ============================================================

def bfs(start, goal):

    # --------------------------------------------------------
    # BFS uses a QUEUE.
    #
    # Queue = First In First Out (FIFO)
    # --------------------------------------------------------

    queue = deque()

    # Put START into queue
    queue.append(start)

    # Keep track of visited states
    visited = {start}

    # Store parent of every state
    parent = {
        start: None
    }

    # Store the move used to reach every state
    move_used = {
        start: None
    }

    # --------------------------------------------------------
    # Continue until queue becomes empty
    # --------------------------------------------------------

    while queue:

        # Remove the FIRST state from queue
        current = queue.popleft()

        # ----------------------------------------------------
        # Goal test
        # ----------------------------------------------------

        if current == goal:

            return reconstruct_path(
                parent,
                move_used,
                goal
            )

        # ----------------------------------------------------
        # Generate all possible next states
        # ----------------------------------------------------

        for next_state, move in get_neighbors(current):

            # Only visit a state once
            if next_state not in visited:

                visited.add(next_state)

                # Remember where it came from
                parent[next_state] = current

                # Remember the move
                move_used[next_state] = move

                # Add state to END of queue
                queue.append(next_state)

    # No solution
    return None


# ============================================================
# DFS - DEPTH FIRST SEARCH
# ============================================================

def dfs(start, goal):

    # --------------------------------------------------------
    # DFS uses a STACK.
    #
    # Stack = Last In First Out (LIFO)
    # --------------------------------------------------------

    stack = []

    # Put START into stack
    stack.append(start)

    # Keep track of visited states
    visited = set()

    # Store parent information
    parent = {
        start: None
    }

    # Store move information
    move_used = {
        start: None
    }

    # --------------------------------------------------------
    # Continue until stack becomes empty
    # --------------------------------------------------------

    while stack:

        # Remove the LAST state from stack
        current = stack.pop()

        # If already visited, skip it
        if current in visited:
            continue


        # Mark as visited
        visited.add(current)

        # ----------------------------------------------------
        # Goal test
        # ----------------------------------------------------

        if current == goal:

            return reconstruct_path(
                parent,
                move_used,
                goal
            )

        # ----------------------------------------------------
        # Generate possible moves
        # ----------------------------------------------------

        neighbors = get_neighbors(current)

        # ----------------------------------------------------
        # Add children to stack.
        #
        # We reverse the list so that the first generated
        # child is explored first because STACK is LIFO.
        # ----------------------------------------------------

        for next_state, move in reversed(neighbors):

            if next_state not in visited:

                # Store parent only the first time
                if next_state not in parent:

                    parent[next_state] = current

                    move_used[next_state] = move

                # Put child into stack
                stack.append(next_state)

    # No solution
    return None


# ============================================================
# DEPTH LIMITED SEARCH
# Used by IDS
# ============================================================

def depth_limited_search(
        current,
        goal,
        depth_limit,
        path,
        visited
    ):
    """
    DFS with a maximum depth.

    IDS repeatedly calls this function with:

    depth = 0
    depth = 1
    depth = 2
    depth = 3
    ...
    """

    # --------------------------------------------------------
    # Goal test
    # --------------------------------------------------------

    if current == goal:
        return path

    # --------------------------------------------------------
    # If depth limit is 0,
    # we cannot go any deeper.
    # --------------------------------------------------------

    if depth_limit == 0:
        return None

    # Add current state to current DFS path
    visited.add(current)

    # --------------------------------------------------------
    # Generate children
    # --------------------------------------------------------

    for next_state, move in get_neighbors(current):

        # Avoid going back to a state on the current path
        if next_state not in visited:

            # Add child to path
            path.append(
                (next_state, move)
            )

            # Search child with one less depth available
            result = depth_limited_search(
                next_state,
                goal,
                depth_limit - 1,
                path,
                visited
            )

            # If goal was found
            if result is not None:
                return result

            # ------------------------------------------------
            # Backtrack

            # Remove child because this branch failed.
            # ------------------------------------------------

            path.pop()

    # Remove current state while backtracking
    visited.remove(current)

    return None


# ============================================================
# IDS - ITERATIVE DEEPENING SEARCH
# ============================================================

def ids(start, goal):

    # Start with depth limit 0
    depth_limit = 0

    # --------------------------------------------------------
    # Keep increasing depth limit
    # --------------------------------------------------------

    while True:

        # New path starts from START every time
        path = [
            (start, None)
        ]

        visited = set()

        result = depth_limited_search(start, goal, depth_limit, path, visited)

        if result is not None:
            return result

        # If not found, increase depth limit
        depth_limit += 1


In [2]:
# ============================================================
# RUNNING BFS (BREADTH-FIRST SEARCH)
# ============================================================
print("--- RUNNING BFS ---")
bfs_path = bfs(START, GOAL)

if bfs_path:
    print(f"Solvable in {len(bfs_path) - 1} steps.\n")
    for step_num, (state, move) in enumerate(bfs_path):
        if move:
            print(f"Step {step_num}: Move {move}")
        else:
            print("Start State:")
        print_state(state)
else:
    print("No solution found.")

--- RUNNING BFS ---
Solvable in 24 steps.

Start State:
5 4 □
6 1 8
7 3 2

Step 1: Move LEFT
5 □ 4
6 1 8
7 3 2

Step 2: Move DOWN
5 1 4
6 □ 8
7 3 2

Step 3: Move DOWN
5 1 4
6 3 8
7 □ 2

Step 4: Move LEFT
5 1 4
6 3 8
□ 7 2

Step 5: Move UP
5 1 4
□ 3 8
6 7 2

Step 6: Move RIGHT
5 1 4
3 □ 8
6 7 2

Step 7: Move RIGHT
5 1 4
3 8 □
6 7 2

Step 8: Move DOWN
5 1 4
3 8 2
6 7 □

Step 9: Move LEFT
5 1 4
3 8 2
6 □ 7

Step 10: Move LEFT
5 1 4
3 8 2
□ 6 7

Step 11: Move UP
5 1 4
□ 8 2
3 6 7

Step 12: Move UP
□ 1 4
5 8 2
3 6 7

Step 13: Move RIGHT
1 □ 4
5 8 2
3 6 7

Step 14: Move RIGHT
1 4 □
5 8 2
3 6 7

Step 15: Move DOWN
1 4 2
5 8 □
3 6 7

Step 16: Move LEFT
1 4 2
5 □ 8
3 6 7

Step 17: Move LEFT
1 4 2
□ 5 8
3 6 7

Step 18: Move DOWN
1 4 2
3 5 8
□ 6 7

Step 19: Move RIGHT
1 4 2
3 5 8
6 □ 7

Step 20: Move RIGHT
1 4 2
3 5 8
6 7 □

Step 21: Move UP
1 4 2
3 5 □
6 7 8

Step 22: Move LEFT
1 4 2
3 □ 5
6 7 8

Step 23: Move UP
1 □ 2
3 4 5
6 7 8

Step 24: Move LEFT
□ 1 2
3 4 5
6 7 8



In [3]:
# ============================================================
# RUNNING DFS (DEPTH-FIRST SEARCH)
# ============================================================
print("--- RUNNING DFS ---")
dfs_path = dfs(START, GOAL)

if dfs_path:
    print(f"Solvable in {len(dfs_path) - 1} steps.\n")
    # Only printing the first few steps and final result to avoid excessive output
    print("First state:")
    print_state(dfs_path[0][0])
    print(f"... [{len(dfs_path) - 2} intermediate transitions] ...\n")
    print("Goal state:")
    print_state(dfs_path[-1][0])
else:
    print("No solution found.")

--- RUNNING DFS ---
Solvable in 688 steps.

First state:
5 4 □
6 1 8
7 3 2

... [687 intermediate transitions] ...

Goal state:
□ 1 2
3 4 5
6 7 8



In [4]:
# ============================================================
# RUNNING IDS (ITERATIVE DEEPENING SEARCH)
# ============================================================
print("--- RUNNING IDS ---")
ids_path = ids(START, GOAL)

if ids_path:
    print(f"Solvable in {len(ids_path) - 1} steps.\n")
    for step_num, (state, move) in enumerate(ids_path):
        if move:
            print(f"Step {step_num}: Move {move}")
        else:
            print("Start State:")
        print_state(state)
else:
    print("No solution found.")

--- RUNNING IDS ---
Solvable in 24 steps.

Start State:
5 4 □
6 1 8
7 3 2

Step 1: Move LEFT
5 □ 4
6 1 8
7 3 2

Step 2: Move DOWN
5 1 4
6 □ 8
7 3 2

Step 3: Move DOWN
5 1 4
6 3 8
7 □ 2

Step 4: Move LEFT
5 1 4
6 3 8
□ 7 2

Step 5: Move UP
5 1 4
□ 3 8
6 7 2

Step 6: Move RIGHT
5 1 4
3 □ 8
6 7 2

Step 7: Move RIGHT
5 1 4
3 8 □
6 7 2

Step 8: Move DOWN
5 1 4
3 8 2
6 7 □

Step 9: Move LEFT
5 1 4
3 8 2
6 □ 7

Step 10: Move LEFT
5 1 4
3 8 2
□ 6 7

Step 11: Move UP
5 1 4
□ 8 2
3 6 7

Step 12: Move UP
□ 1 4
5 8 2
3 6 7

Step 13: Move RIGHT
1 □ 4
5 8 2
3 6 7

Step 14: Move RIGHT
1 4 □
5 8 2
3 6 7

Step 15: Move DOWN
1 4 2
5 8 □
3 6 7

Step 16: Move LEFT
1 4 2
5 □ 8
3 6 7

Step 17: Move LEFT
1 4 2
□ 5 8
3 6 7

Step 18: Move DOWN
1 4 2
3 5 8
□ 6 7

Step 19: Move RIGHT
1 4 2
3 5 8
6 □ 7

Step 20: Move RIGHT
1 4 2
3 5 8
6 7 □

Step 21: Move UP
1 4 2
3 5 □
6 7 8

Step 22: Move LEFT
1 4 2
3 □ 5
6 7 8

Step 23: Move UP
1 □ 2
3 4 5
6 7 8

Step 24: Move LEFT
□ 1 2
3 4 5
6 7 8

